# AUC 改善实验台 v2.2（特征消融 + 时间切分CV + PSI漂移剔除 + 多候选提交）
本笔记本旨在**系统地**做多方案消融与稳健性评估，并一次性产出可对比的结果与候选提交文件，避免线上提交次数受限时“盲投”。

## 功能概览
1. 统一加载并合并主表与流水特征（v2.1 版本）。
2. **时间切分 CV**（按 `record_time` 分桶）与常规 5 折分层 CV 并行评估。
3. **PSI 漂移检测**（train vs test，如果有测试集；若无则早期 vs 晚期训练集）：自动建议剔除高漂移特征。
4. 定义多组**消融方案**（如移除高漂移高基数列 / 移除部分流水统计 / 调整 EWM 半衰期 / 切换类别权重策略）。
5. 训练 CatBoost（带更稳参数），记录 **OOF AUC、TimeSplit AUC**、特征重要性、OOF 预测。
6. 若有测试集，产出多份候选 `submission_*.csv`，并生成 `experiments_summary.csv` 汇总对比。


In [1]:
# 依赖（环境未装时可解注安装）
# !pip install catboost scikit-learn pandas numpy -q
import os, json, math, gc, itertools
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import StratifiedKFold
from catboost import CatBoostClassifier, Pool
SEED = 2025
np.random.seed(SEED)
SECONDS_PER_DAY = 86400.0
NON_FEATURE_COLS = ['id','label']
def to_dt(series):
    return pd.to_datetime(pd.to_numeric(series, errors='coerce'), unit='s', utc=True)


In [2]:
# 路径配置
TRAIN_CSV = 'train/train.csv'
TRAIN_STMT_FEAT = 'train/train_statement_feature_v2.csv'
TEST_CSV = 'testaa/testaa.csv'                      # 如果没有可置为 None
TEST_STMT_FEAT = 'testaa/testaa_statement_feature_v2.csv'  # 如果没有可置为 None
OUT_SUBMISSION = 'output_Cat/submission.csv'
SAVE_OOF = 'output_Cat/oof_pred.csv'
SAVE_INFO = 'output_Cat/cv_info.json'

OUT_DIR = 'output_CAT/exp_v22_out'
os.makedirs(OUT_DIR, exist_ok=True)


In [3]:
# 数据加载与合并
train = pd.read_csv(TRAIN_CSV)
stmt = pd.read_csv(TRAIN_STMT_FEAT)
df = train.merge(stmt, on='id', how='left', suffixes=('', '_stmt'))
df['stmt_missing'] = df['has_statement'].apply(lambda x: 0 if x==1 else 1) if 'has_statement' in df.columns else 1
y = df['label'].astype(int)
X_all = df.drop(columns=['label']).copy()

X_test = None
if TEST_CSV is not None and os.path.exists(TEST_CSV):
    test = pd.read_csv(TEST_CSV)
    if TEST_STMT_FEAT is not None and os.path.exists(TEST_STMT_FEAT):
        stmt_t = pd.read_csv(TEST_STMT_FEAT)
        X_test = test.merge(stmt_t, on='id', how='left', suffixes=('', '_stmt'))
        X_test['stmt_missing'] = X_test['has_statement'].apply(lambda x: 0 if x==1 else 1) if 'has_statement' in X_test.columns else 1
    else:
        X_test = test.copy(); X_test['stmt_missing'] = 1
print('训练样本：', X_all.shape, '测试样本：', None if X_test is None else X_test.shape)


训练样本： (53480, 41) 测试样本： (20054, 41)


In [4]:
# 基础特征工程（与 v2.1 一致，并增加 Winsorize 可选开关）
from pandas.api.types import is_numeric_dtype

def _coerce_numeric(series):
    return pd.to_numeric(series, errors='coerce')

def _safe_div(a,b):
    try:
        b = b.replace(0, np.nan)
    except Exception:
        b = np.where(b==0, np.nan, b)
    return a / b

def winsorize_df(X: pd.DataFrame, lower=0.01, upper=0.99):
    Xw = X.copy()
    for c in Xw.columns:
        if c in NON_FEATURE_COLS: continue
        if is_numeric_dtype(Xw[c]):
            ql, qu = Xw[c].quantile(lower), Xw[c].quantile(upper)
            Xw[c] = Xw[c].clip(ql, qu)
    return Xw

def base_feature_engineering(df: pd.DataFrame, do_winsor=False):
    if df is None: return None
    X = df.copy()
    # 清洗字符串缺失
    for col in X.columns:
        if X[col].dtype == 'O':
            X[col] = X[col].replace(['', ' ', 'nan', 'NaN', 'NULL', 'None'], np.nan)

    # level 解析
    if 'level' in X.columns:
        X['level'] = X['level'].astype(str)
        X['grade'] = X['level'].str[0]
        X['subgrade'] = pd.to_numeric(X['level'].str[1:], errors='coerce')
        X['grade_rank'] = X['grade'].map({'A':5,'B':4,'C':3,'D':2,'E':1}).fillna(0).astype('Int64')

    # 日期化相对时间
    if {'record_time','issue_time'}.issubset(X.columns):
        rec = to_dt(X['record_time']); iss = to_dt(X['issue_time'])
        X['days_issue_to_record'] = ((rec - iss).dt.total_seconds()/SECONDS_PER_DAY).clip(lower=0)
    if {'record_time','history_time'}.issubset(X.columns):
        rec = to_dt(X['record_time']); his = to_dt(X['history_time'])
        X['days_history_to_record'] = ((rec - his).dt.total_seconds()/SECONDS_PER_DAY).clip(lower=0)

    # 比率/规模
    if {'balance','balance_limit'}.issubset(X.columns):
        X['balance_utilization'] = _safe_div(_coerce_numeric(X['balance']), _coerce_numeric(X['balance_limit']))
        X['balance_utilization_sqrt'] = np.sqrt(X['balance_utilization'].clip(lower=0))
    if {'balance_accounts','total_accounts'}.issubset(X.columns):
        X['acct_utilization'] = _safe_div(_coerce_numeric(X['balance_accounts']), _coerce_numeric(X['total_accounts']))
    if {'loan','balance_limit'}.issubset(X.columns):
        X['loan_to_limit'] = _safe_div(_coerce_numeric(X['loan']), _coerce_numeric(X['balance_limit']))
    if {'loan','term'}.issubset(X.columns):
        X['loan_per_month'] = _safe_div(_coerce_numeric(X['loan']), _coerce_numeric(X['term']))

    # 明显枚举作为类别
    for c in ['title','career','zip_code','residence','term','syndicated','installment','level','grade']:
        if c in X.columns:
            if c in ['level','grade']:
                X[c] = X[c].astype('object')
            else:
                X[c] = pd.to_numeric(X[c], errors='coerce').astype('Int64')

    if do_winsor:
        X = winsorize_df(X)

    # 删原始时间戳
    for col in ['issue_time','record_time','history_time']:
        if col in X.columns:
            X.drop(columns=[col], inplace=True)
    return X

def get_cat_cols(df: pd.DataFrame):
    cat_cols = []
    for c in ['title','career','zip_code','residence','term','syndicated','installment','level','grade']:
        if c in df.columns:
            cat_cols.append(c)
    for c in df.select_dtypes(include='object').columns:
        if c not in cat_cols:
            cat_cols.append(c)
    cat_cols = [c for c in cat_cols if c not in NON_FEATURE_COLS and c in df.columns]
    return sorted(list(set(cat_cols)))

def fill_missing(df: pd.DataFrame, cat_cols):
    from pandas.api.types import is_numeric_dtype
    X = df.copy()
    for c in X.columns:
        if c in NON_FEATURE_COLS: continue
        if c in cat_cols:
            X[c] = X[c].astype('object').fillna('Unknown'); continue
        if is_numeric_dtype(X[c]):
            X[c] = X[c].fillna(X[c].median()); continue
        xnum = pd.to_numeric(X[c], errors='coerce')
        if xnum.notna().any():
            X[c] = xnum.fillna(xnum.median())
        else:
            X[c] = X[c].astype('object').fillna('Unknown')
    return X

def ensure_catboost_dtypes(df: pd.DataFrame, cat_cols):
    X = df.copy()
    for c in cat_cols:
        if c in X.columns:
            X[c] = X[c].astype(str)
    for c in X.columns:
        if c in cat_cols or c in NON_FEATURE_COLS: continue
        X[c] = pd.to_numeric(X[c], errors='coerce').astype('float32')
    return X


In [5]:
# PSI 计算：数值按分位数分箱，类别按频次分布
def psi_numeric(train_s, test_s, bins=10):
    t = pd.to_numeric(train_s, errors='coerce'); q = t.quantile(np.linspace(0,1,bins+1)).values
    q = np.unique(q)
    if len(q) <= 2:
        return 0.0
    t_bin = pd.cut(t, bins=q, include_lowest=True)
    s_bin = pd.cut(pd.to_numeric(test_s, errors='coerce'), bins=q, include_lowest=True)
    t_p = t_bin.value_counts(normalize=True); s_p = s_bin.value_counts(normalize=True)
    idx = t_p.index.union(s_p.index)
    t_p = t_p.reindex(idx).fillna(1e-6); s_p = s_p.reindex(idx).fillna(1e-6)
    return float(((t_p - s_p) * np.log(t_p / s_p)).sum())

def psi_categorical(train_s, test_s):
    t_p = train_s.astype(str).value_counts(normalize=True)
    s_p = test_s.astype(str).value_counts(normalize=True)
    idx = t_p.index.union(s_p.index)
    t_p = t_p.reindex(idx).fillna(1e-6); s_p = s_p.reindex(idx).fillna(1e-6)
    return float(((t_p - s_p) * np.log(t_p / s_p)).sum())

def compute_psi_report(X_train_full, X_test_full=None, record_time_train=None):
    report = []
    # 如果没有测试集，就用时间切分：早 50% vs 晚 50%
    if X_test_full is None:
        assert record_time_train is not None, '缺少 record_time 以做时间替代 PSI.'
        dt = to_dt(record_time_train)
        cutoff = dt.quantile(0.5)
        left_idx = dt <= cutoff
        right_idx = dt > cutoff
        A, B = X_train_full[left_idx], X_train_full[right_idx]
    else:
        A, B = X_train_full, X_test_full

    for c in A.columns:
        if c in NON_FEATURE_COLS: continue
        try:
            if pd.api.types.is_numeric_dtype(A[c]):
                psi = psi_numeric(A[c], B[c])
            else:
                psi = psi_categorical(A[c], B[c])
        except Exception:
            psi = np.nan
        report.append({'feature': c, 'psi': psi})
    rep = pd.DataFrame(report).sort_values('psi', ascending=False)
    return rep


In [6]:
# 时间切分 KFold（按 record_time 排序后等份切割）
def time_split_indices(record_time_series, n_splits=5):
    dt = to_dt(record_time_series)
    order = np.argsort(dt.values)
    idx = np.arange(len(dt))[order]
    folds = np.array_split(idx, n_splits)
    return folds  # list of index arrays from earliest to latest


In [7]:
# CatBoost 训练/评估通用函数
def run_catboost_cv(X, y, cat_cols, params, n_splits=5, seed=SEED, cv_type='skf'):
    if cv_type == 'skf':
        splitter = list(StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed).split(X, y))
    else:
        # time-based: folds是按时间顺序的块；我们用前k-1块训练，第k块验证（滚动窗口）
        folds = time_split_indices(original['record_time'])
        splitter = []
        for k in range(1, n_splits):
            train_idx = np.concatenate(folds[:k])
            val_idx = folds[k]
            splitter.append((train_idx, val_idx))

    feat_cols = [c for c in X.columns if c not in NON_FEATURE_COLS]
    cat_idx = [feat_cols.index(c) for c in cat_cols if c in feat_cols]

    oof = np.zeros(len(X))
    best_iterations = []
    fold_auc = []

    for fold, (tr_idx, va_idx) in enumerate(splitter, 1):
        X_tr, X_va = X.iloc[tr_idx][feat_cols], X.iloc[va_idx][feat_cols]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        model = CatBoostClassifier(**params)
        model.fit(Pool(X_tr, label=y_tr, cat_features=cat_idx), eval_set=Pool(X_va, label=y_va, cat_features=cat_idx), verbose=False)
        va_pred = model.predict_proba(X_va)[:,1]
        oof[va_idx] = va_pred
        fold_auc.append(roc_auc_score(y_va, va_pred))
        best_iterations.append(model.get_best_iteration())
    return oof, fold_auc, best_iterations


In [8]:
# 统一预处理管线（功能：特征工程->填充->类型统一->剔除列）
def prepare_matrix(df, drop_cols=None, do_winsor=False):
    X = base_feature_engineering(df, do_winsor=do_winsor)
    if drop_cols:
        drop_cols_eff = [c for c in drop_cols if c in X.columns]
        if drop_cols_eff:
            X = X.drop(columns=drop_cols_eff)
    cat_cols = get_cat_cols(X)
    X = fill_missing(X, cat_cols)
    X = ensure_catboost_dtypes(X, cat_cols)
    return X, cat_cols


In [9]:
# 默认参数（稳健）
def default_params(seed=SEED):
    return dict(
        loss_function='Logloss', eval_metric='AUC',
        iterations=8000, learning_rate=0.02, depth=6, l2_leaf_reg=10.0,
        random_seed=seed, bootstrap_type='Bayesian', bagging_temperature=0.5,
        rsm=0.8, random_strength=2.0, border_count=128, grow_policy='SymmetricTree',
        od_type='Iter', od_wait=400, verbose=False, allow_writing_files=False,
        auto_class_weights='Balanced'
    )


In [10]:
# 漂移检测（PSI）并保存报告
original = train.merge(stmt, on='id', how='left', suffixes=('', '_stmt'))
psi_report_path = os.path.join(OUT_DIR, 'psi_report.csv')
psi_rep = compute_psi_report(X_all, X_test, record_time_train=original['record_time'])
psi_rep.to_csv(psi_report_path, index=False)
print('PSI 报告保存：', psi_report_path)

# 自动建议剔除 PSI 过高的列（>0.25）
PSI_CUT = 0.25
suggest_drop_by_psi = psi_rep.loc[psi_rep['psi'] > PSI_CUT, 'feature'].tolist()
with open(os.path.join(OUT_DIR, 'suggest_drop_by_psi.txt'), 'w', encoding='utf-8') as f:
    f.write('\n'.join(suggest_drop_by_psi))
print('建议按漂移剔除的列数：', len(suggest_drop_by_psi))


PSI 报告保存： output_CAT/exp_v22_out/psi_report.csv
建议按漂移剔除的列数： 5


In [11]:
# 定义一组实验配置（可自行增删）
EXPS = []
# BaseA：去高漂移高基数（zip_code、title、residence），Winsorize 数值，使用Balanced权重
EXPS.append(dict(name='A_base_dropgeo_winz', drop_cols=['zip_code','title','residence'], do_winsor=True, use_scale_pos_weight=False))
# B：在A基础上，去掉部分流水统计（span_days/active_days/tx_count），保留EWM
EXPS.append(dict(name='B_drop_stmt_counts_keep_ewm', drop_cols=['zip_code','title','residence','span_days','active_days','tx_count'], do_winsor=True, use_scale_pos_weight=False))
# C：A + 用 scale_pos_weight 替换 Balanced（互斥）
EXPS.append(dict(name='C_base_scale_pos_weight', drop_cols=['zip_code','title','residence'], do_winsor=True, use_scale_pos_weight=True))
# D：A + 按 PSI 自动剔除
EXPS.append(dict(name='D_base_auto_psi', drop_cols=['zip_code','title','residence'] + suggest_drop_by_psi, do_winsor=True, use_scale_pos_weight=False))
# E：A + 去掉所有 EWM 相关特征
EXPS.append(dict(name='E_base_drop_ewm', drop_cols=['zip_code','title','residence','ewm_income','ewm_expense_abs','ewm_tx','ewm_income_expense_ratio'], do_winsor=True, use_scale_pos_weight=False))

pd.DataFrame(EXPS).to_csv(os.path.join(OUT_DIR, 'experiment_configs.csv'), index=False)
print('已定义实验方案：', [e['name'] for e in EXPS])


已定义实验方案： ['A_base_dropgeo_winz', 'B_drop_stmt_counts_keep_ewm', 'C_base_scale_pos_weight', 'D_base_auto_psi', 'E_base_drop_ewm']


In [12]:
# 运行实验
summary_rows = []
for exp in EXPS:
    name = exp['name']
    drop_cols = exp.get('drop_cols', [])
    do_winsor = exp.get('do_winsor', False)
    use_scale_pos_weight = exp.get('use_scale_pos_weight', False)

    # 准备特征矩阵
    X_fe, cat_cols = prepare_matrix(X_all, drop_cols=drop_cols, do_winsor=do_winsor)
    # 备份 feature list
    feat_cols = [c for c in X_fe.columns if c not in NON_FEATURE_COLS]
    with open(os.path.join(OUT_DIR, f'features_{name}.txt'), 'w', encoding='utf-8') as f:
        f.write('\n'.join(feat_cols))

    # 参数设置
    params = default_params(seed=SEED)
    if use_scale_pos_weight:
        pos = int((y==1).sum()); neg = int((y==0).sum())
        params.pop('auto_class_weights', None)
        params['scale_pos_weight'] = max(1.0, neg/max(1,pos))

    # 5折分层CV
    oof_skf, fold_auc_skf, best_iter_skf = run_catboost_cv(X_fe, y, cat_cols, params, n_splits=5, seed=SEED, cv_type='skf')
    cv_auc_skf = roc_auc_score(y, oof_skf)
    pd.DataFrame({'id': X_all['id'], 'oof_pred': oof_skf, 'label': y}).to_csv(os.path.join(OUT_DIR, f'oof_{name}.csv'), index=False)

    # 时间切分CV
    oof_time, fold_auc_time, best_iter_time = run_catboost_cv(X_fe, y, cat_cols, params, n_splits=5, seed=SEED, cv_type='time')
    cv_auc_time = roc_auc_score(y, oof_time)
    pd.DataFrame({'id': X_all['id'], 'oof_pred': oof_time, 'label': y}).to_csv(os.path.join(OUT_DIR, f'oof_time_{name}.csv'), index=False)

    # 保存折分明细
    with open(os.path.join(OUT_DIR, f'cv_{name}.json'),'w',encoding='utf-8') as f:
        json.dump({'cv_auc_skf': float(cv_auc_skf), 'fold_auc_skf': [float(x) for x in fold_auc_skf],
                   'cv_auc_time': float(cv_auc_time), 'fold_auc_time': [float(x) for x in fold_auc_time],
                   'best_iter_skf': [int(x) for x in best_iter_skf], 'best_iter_time': [int(x) for x in best_iter_time]
                  }, f, ensure_ascii=False, indent=2)

    # 特征重要性（用全量复训一遍快速计算）
    feat_cols = [c for c in X_fe.columns if c not in NON_FEATURE_COLS]
    cat_idx = [feat_cols.index(c) for c in cat_cols if c in feat_cols]
    model = CatBoostClassifier(**{**params, 'iterations': max(200,int(np.mean(best_iter_skf)))})
    model.fit(Pool(X_fe[feat_cols], label=y, cat_features=cat_idx), verbose=False)
    imp = model.get_feature_importance(Pool(X_fe[feat_cols], label=y, cat_features=cat_idx), type='PredictionValuesChange')
    pd.DataFrame({'feature': feat_cols, 'importance': imp}).sort_values('importance', ascending=False).to_csv(
        os.path.join(OUT_DIR, f'feature_importance_{name}.csv'), index=False)

    # 如果有测试集，产出候选提交
    sub_path = None
    if X_test is not None:
        X_test_fe, _ = prepare_matrix(X_test, drop_cols=drop_cols, do_winsor=do_winsor)
        preds = model.predict_proba(X_test_fe[feat_cols])[:,1]
        sub = pd.DataFrame({'id': X_test['id'].values, 'prob': preds})
        sub_path = os.path.join(OUT_DIR, f'submission_{name}.csv')
        sub.to_csv(sub_path, index=False)

    summary_rows.append(dict(
        name=name,
        cv_auc_skf=float(cv_auc_skf),
        cv_auc_time=float(cv_auc_time),
        skf_folds=[float(x) for x in fold_auc_skf],
        time_folds=[float(x) for x in fold_auc_time],
        features=len(feat_cols),
        drop_cols=drop_cols,
        use_scale_pos_weight=use_scale_pos_weight,
        submission=sub_path
    ))
    print(f"完成实验 {name}: skf={cv_auc_skf:.4f}, time={cv_auc_time:.4f}, 提交={sub_path}")
    gc.collect()

summary = pd.DataFrame(summary_rows)
summary_path = os.path.join(OUT_DIR, 'experiments_summary.csv')
summary.to_csv(summary_path, index=False)
summary


完成实验 A_base_dropgeo_winz: skf=0.6380, time=0.6038, 提交=output_CAT/exp_v22_out/submission_A_base_dropgeo_winz.csv
完成实验 B_drop_stmt_counts_keep_ewm: skf=0.6392, time=0.6039, 提交=output_CAT/exp_v22_out/submission_B_drop_stmt_counts_keep_ewm.csv
完成实验 C_base_scale_pos_weight: skf=0.6381, time=0.6072, 提交=output_CAT/exp_v22_out/submission_C_base_scale_pos_weight.csv
完成实验 D_base_auto_psi: skf=0.6400, time=0.6064, 提交=output_CAT/exp_v22_out/submission_D_base_auto_psi.csv
完成实验 E_base_drop_ewm: skf=0.6398, time=0.6029, 提交=output_CAT/exp_v22_out/submission_E_base_drop_ewm.csv


,name,cv_auc_skf,cv_auc_time,skf_folds,time_folds,features,drop_cols,use_scale_pos_weight,submission
0,A_base_dropgeo_winz,0.638040,0.603793,"[0.6379833273608766, 0.6423461503586694, 0.634...","[0.5686715647688084, 0.5772769247945451, 0.650...",44,"[zip_code, title, residence]",False,output_CAT/exp_v22_out/submission_A_base_dropg...
1,B_drop_stmt_counts_keep_ewm,0.639207,0.603927,"[0.6375646462591391, 0.6439871691095145, 0.632...","[0.572076231219251, 0.5847586107616383, 0.6508...",41,"[zip_code, title, residence, span_days, active...",False,output_CAT/exp_v22_out/submission_B_drop_stmt_...
2,C_base_scale_pos_weight,0.638083,0.607176,"[0.637508663542889, 0.6432662973663815, 0.6333...","[0.5721882448526892, 0.5812892363712525, 0.647...",44,"[zip_code, title, residence]",True,output_CAT/exp_v22_out/submission_C_base_scale...
3,D_base_auto_psi,0.639964,0.606404,"[0.6397542956292349, 0.6423598414685502, 0.634...","[0.5764942323849552, 0.5785100730426493, 0.647...",39,"[zip_code, title, residence, record_dt, sum_ex...",False,output_CAT/exp_v22_out/submission_D_base_auto_...
4,E_base_drop_ewm,0.639766,0.602941,"[0.6385545831193372, 0.6429025547436998, 0.634...","[0.5719501898215614, 0.581524326441993, 0.6483...",40,"[zip_code, title, residence, ewm_income, ewm_e...",False,output_CAT/exp_v22_out/submission_E_base_drop_...
